In [ ]:
#mount drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/path/to/folder')

In [ ]:
!pip install segmentation-models-pytorch albumentations -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.6 MB/s eta 0:00:00


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset
import cv2
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.models.segmentation import fcn_resnet50
import torch.nn as nn
import segmentation_models_pytorch as smp
import numpy as np
import shutil

ModuleNotFoundError: No module named 'segmentation_models_pytorch'

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/')

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
train_image_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/CVAT_001_first_batch/CVAT_001_first_images")
train_mask_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/CVAT_001_first_batch/CVAT_001_first_masks")
val_image_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/val/val_images")
val_mask_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/val/val_masks")
batch2_image_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/CVAT_001_Second_Batch_Images")

U_NET_export_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/Second Batch: UNET/1st200_masks")


In [ ]:

mask_dir = Path(U_NET_export_dir)

fixed_mask_dir = Path("/content/fixed_lichen_masks")
fixed_mask_dir.mkdir(parents=True, exist_ok=True)

for mask_path in sorted(mask_dir.glob("*.png")):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    fixed = (mask > 0).astype(np.uint8)  # converts 255 → 1

    cv2.imwrite(str(fixed_mask_dir / mask_path.name), fixed)

print("Fixed masks:", len(list(fixed_mask_dir.glob('*.png'))))

Fixed masks: 200


In [ ]:

mask_dir = fixed_mask_dir
voc_root = Path("/content/lichen_voc_upload")

# clean old version
if voc_root.exists():
    shutil.rmtree(voc_root)

seg_class_dir = voc_root / "SegmentationClass"
image_sets_dir = voc_root / "ImageSets" / "Segmentation"

seg_class_dir.mkdir(parents=True, exist_ok=True)
image_sets_dir.mkdir(parents=True, exist_ok=True)

mask_files = sorted(mask_dir.glob("*.png"))

# Copy masks into SegmentationClass
stems = []

for mask_path in mask_files:
    stem = mask_path.stem
    stems.append(stem)

    shutil.copy(
        mask_path,
        seg_class_dir / mask_path.name
    )

# Create required txt file
with open(image_sets_dir / "default.txt", "w") as f:
    for stem in stems:
        f.write(stem + "\n")

# Zip the CONTENTS of voc_root
zip_path = shutil.make_archive(
    "/content/lichen_voc_upload",
    "zip",
    voc_root
)

print("Created:")
print(zip_path)
print("Masks:", len(mask_files))

Created:
/content/lichen_voc_upload.zip
Masks: 200


In [ ]:


image_dir = Path(batch2_image_dir)
mask_dir = Path(U_NET_export_dir)

out_json = "/content/lichen_coco_predictions.json"

images = []
annotations = []
categories = [
    {"id": 1, "name": "lichen", "supercategory": "object"}
]

ann_id = 1

image_files = sorted([
    p for p in image_dir.rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

for img_id, img_path in enumerate(image_files, start=1):
    mask_path = mask_dir / (img_path.stem + ".png")

    if not mask_path.exists():
        print("Missing mask:", img_path.name)
        continue

    with Image.open(img_path) as im:
        width, height = im.size

    images.append({
        "id": img_id,
        "file_name": img_path.name,
        "width": width,
        "height": height
    })

    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    mask = (mask > 0).astype(np.uint8)

    if mask.shape[:2] != (height, width):
        mask = cv2.resize(
            mask,
            (width, height),
            interpolation=cv2.INTER_NEAREST
        )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for contour in contours:
        if len(contour) < 3:
            continue

        segmentation = contour.reshape(-1, 2).flatten().astype(float).tolist()

        if len(segmentation) < 6:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        area = float(cv2.contourArea(contour))

        if area < 10:
            continue

        annotations.append({
            "id": ann_id,
            "image_id": img_id,
            "category_id": 1,
            "segmentation": [segmentation],
            "area": area,
            "bbox": [float(x), float(y), float(w), float(h)],
            "iscrowd": 0
        })

        ann_id += 1

coco = {
    "images": images,
    "annotations": annotations,
    "categories": categories
}

with open(out_json, "w") as f:
    json.dump(coco, f)

print("Saved COCO annotations:")
print(out_json)
print("Images:", len(images))
print("Annotations:", len(annotations))

Saved COCO annotations:
/content/lichen_coco_predictions.json
Images: 200
Annotations: 4891


In [ ]:
import zipfile

zip_path = "/content/lichen_coco_predictions.zip"

with zipfile.ZipFile(zip_path, "w") as z:
    z.write(
        "/content/lichen_coco_predictions.json",
        arcname="annotations/instances_default.json"
    )

print(zip_path)

/content/lichen_coco_predictions.zip


In [ ]:
!ls /content

drive		    lichen_voc_upload	   sample_data
fixed_lichen_masks  lichen_voc_upload.zip


In [ ]:
from google.colab import files

files.download("/content/lichen_coco_predictions.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls "/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/utils"

best_quandrant.py      find_matching_image.py  lichen_dataset.py  __pycache__
comparison_metrics.py  get_box.py	       pred_quadrant.py   training.py


In [ ]:
from lichen_dataset import LichenDataset

ImportError: cannot import name 'LichenDataset' from 'lichen_dataset' (/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/utils/lichen_dataset.py)

In [ ]:
from utils.lichen_dataset import *

In [ ]:
#Dataset and Image Loaders


IMAGE_SIZE = (512, 512)
BATCH_SIZE = 4

train_dataset = LichenDataset(train_image_dir, train_mask_dir)
img, mask = train_dataset[0]


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dataset = LichenDataset(val_image_dir, val_mask_dir)
img, mask = val_dataset[0]

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)



⚠️ Missing mask for: lichen_133244390.jpeg
⚠️ Missing mask for: lichen_33879765.jpg
✅ Found items: 148
⚠️ Missing mask for: lichen_192189784.jpg
⚠️ Missing mask for: lichen_325068990.jpg
✅ Found items: 166


In [ ]:
#U_Net

#encoder = resnet34

unet_resnet34 = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

optimizer_unet_resnet34 = torch.optim.Adam(
    unet_resnet34.parameters(),
    lr=1e-4
)

UNET_RESNET34_SAVE_PATH = (
    f"{U_NET_export_dir}/unet_resnet34_lichen.pth"
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [ ]:
def generate_masks_for_model(
    model,
    checkpoint_path,
    predict_loader,
    output_dir,
    threshold=0.5,
    device="cuda"
):
    model = model.to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        for image, filename, original_h, original_w in predict_loader:
            image = image.to(device)

            logits = model(image)
            probs = torch.sigmoid(logits)

            mask = probs.squeeze().cpu().numpy()
            mask = (mask > threshold).astype("uint8") * 255

            # resize back to original image size
            mask = cv2.resize(
                mask,
                (int(original_w[0]), int(original_h[0])),
                interpolation=cv2.INTER_NEAREST
            )

            mask_name = Path(filename[0]).stem + ".png"
            mask_path = output_dir / mask_name

            cv2.imwrite(str(mask_path), mask)

In [ ]:
from utils.training import *

In [ ]:
EPOCHS = 30

history_unet_resnet34 = train_model(
    model=unet_resnet34,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_unet_resnet34,
    save_path=UNET_RESNET34_SAVE_PATH,
    epochs=EPOCHS,
    device=device
)

Epoch 1/30 | Train Loss: 0.5527 | Val Loss: 0.4618
✅ Saved best model to /content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/Second Batch: UNET/weights/unet_resnet34_lichen.pth
Epoch 2/30 | Train Loss: 0.4344 | Val Loss: 0.4219
✅ Saved best model to /content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/Second Batch: UNET/weights/unet_resnet34_lichen.pth
Epoch 3/30 | Train Loss: 0.3684 | Val Loss: 0.4222
Epoch 4/30 | Train Loss: 0.3451 | Val Loss: 0.4081
✅ Saved best model to /content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/Second Batch: UNET/weights/unet_resnet34_lichen.pth
Epoch 5/30 | Train Loss: 0.3147 | Val Loss: 0.3809
✅ Saved best model to /content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/Second Batch: UNET/weights/unet_resnet34_lichen.pth
Epoch 6/30 | Train Loss: 0.2806 | Val Loss: 0.379

In [ ]:

history_df = pd.DataFrame(history_unet_resnet34)

history_csv_path = os.path.join(
    U_NET_export_dir,
    "unet_resnet34_training_history.csv"
)

history_df.to_csv(history_csv_path, index=False)

print("✅ Training history saved to:")
print(history_csv_path)

✅ Training history saved to:
/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/Second Batch: UNET/weights/unet_resnet34_training_history.csv


In [ ]:
model = unet_resnet34

model.load_state_dict(
    torch.load(UNET_RESNET34_SAVE_PATH, map_location=device)
)

model.to(device)
model.eval()


predict_dataset = LichenDataset(
    image_dir=batch2_image_dir,
    mask_dir=None,
)

predict_loader = DataLoader(
    predict_dataset,
    batch_size=1,
    shuffle=False
)

output_dir = U_NET_export_dir
os.makedirs(output_dir, exist_ok=True)
model.load_state_dict(
    torch.load(UNET_RESNET34_SAVE_PATH, map_location=device)
)

model.to(device)
model.eval()

with torch.no_grad():
    for image, filename in predict_loader:
        image = image.to(device)

        logits = model(image)
        probs = torch.sigmoid(logits)

        mask = probs.squeeze().cpu().numpy()
        binary_mask = (mask > 0.5).astype(np.uint8) * 255

        output_name = os.path.splitext(filename[0])[0] + ".png"
        output_path = os.path.join(output_dir, output_name)

        cv2.imwrite(output_path, binary_mask)

✅ Found items: 200


In [ ]:
print(batch2_image_dir)
!ls "$batch2_image_dir"

/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/CVAT_001_Second_Batch_Images
lichen_133637727.jpeg  lichen_141700384.jpeg  lichen_181200530.jpeg
lichen_136270712.jpg   lichen_141700385.jpeg  lichen_181200531.jpeg
lichen_136306844.jpeg  lichen_141700386.jpeg  lichen_181200532.jpeg
lichen_137622702.jpg   lichen_141700389.jpeg  lichen_181200546.jpeg
lichen_139090803.jpg   lichen_141700390.jpeg  lichen_181200549.jpeg
lichen_139090813.jpg   lichen_141700392.jpeg  lichen_181200554.jpeg
lichen_139989366.jpeg  lichen_141700395.jpeg  lichen_181200555.jpeg
lichen_139989371.jpeg  lichen_141700398.jpeg  lichen_181200560.jpeg
lichen_140252194.jpg   lichen_141700399.jpeg  lichen_181200563.jpeg
lichen_140255565.jpeg  lichen_141767557.jpeg  lichen_181200564.jpeg
lichen_140255566.jpeg  lichen_141767560.jpeg  lichen_181200570.jpeg
lichen_140280908.jpeg  lichen_141767561.jpeg  lichen_181200579.jpeg
lichen_140280910.jpeg  lichen_141767564.jpeg  lichen_18